# BTE 413 Final Group Project: hospital stays in New York, 2024
One notebook for the whole project. Run **Runtime > Run all**. It takes about a minute and needs no Google Drive: the data loads from a public link.

| Part | What | Who |
|---|---|---|
| 1 | The data and the four destination groups | everyone |
| 2 | **Regression:** what will a stay cost, known at admission | Ethan and Niko |
| 3 | **Classification:** where does the patient go after the stay | Mac and Jacob |
| 4 | **The link:** does the likely destination help predict cost | both groups |
| 5 | Do the numbers match the slides? | |

Fairness and ethics (Nikki) are in separate notes.

**Fast mode and full mode.** `RUN_FULL = False` (the default) runs only the final models and their scores. Set `RUN_FULL = True` to also rerun every experiment behind them (variable screens, k-NN, neural network, cross-validation, ablation). Full mode took about 4 minutes on a laptop; on a free Colab runtime allow 20 to 30 minutes.

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report, f1_score, mean_absolute_error,
                             median_absolute_error, precision_recall_fscore_support, r2_score)
warnings.filterwarnings("ignore")

RUN_FULL = False   # True also reruns every experiment (about 4 minutes on a laptop, 20 to 30 on free Colab)

# Public link to the cleaned data (NY SPARCS Hospital Inpatient Discharges 2024, de-identified, 200,000 random stays)
DATA_URL = 'https://raw.githubusercontent.com/ethanmooseg-star/bte413-hospital-cost/main/sparcs_2024_clean_sample.csv.gz'

## Part 1. The data and the destination groups
Cleaned by `clean_sparcs_2024.py` (in the same repository). The train and test labels are fixed in the `split` column, so both groups use the same rows.

In [ ]:
CODE_COLS = ["permanent_facility_id", "apr_mdc_code", "apr_drg_code", "ccsr_diagnosis_code", "ccsr_procedure_code", "zip_code"]
df = pd.read_csv(DATA_URL, dtype={c: str for c in CODE_COLS})
print(f"{len(df):,} stays | {df['split'].value_counts().to_dict()}")
assert df.shape == (200000, 27), "Unexpected shape: the data file changed."
print(f"Cost is very skewed: average ${df['total_costs'].mean():,.0f}, middle ${df['total_costs'].median():,.0f}, so we predict the log of cost.")

The classification group's target `disp4` groups the 19 raw discharge destinations into 4 classes.

In [ ]:
GROUPS = {
    "Home": ["Home or Self Care", "Home w/ Home Health Services"],
    "Post-acute facility": ["Skilled Nursing Home", "Inpatient Rehabilitation Facility",
        "Facility w/ Custodial/Supportive Care", "Medicare Cert Long Term Care Hospital",
        "Medicaid Cert Nursing Facility", "Hosp Basd Medicare Approved Swing Bed"],
    "Died or hospice": ["Expired", "Hospice - Home", "Hospice - Medical Facility"],
    "Other": ["Left Against Medical Advice", "Short-term Hospital",
        "Psychiatric Hospital or Unit of Hosp", "Another Type Not Listed",
        "Court/Law Enforcement", "Cancer Center or Childrens Hospital",
        "Federal Health Care Facility", "Critical Access Hospital"],
}
lookup = {raw: g for g, raws in GROUPS.items() for raw in raws}
df["disp4"] = df["patient_disposition"].map(lookup)
print("unmapped rows:", df["disp4"].isna().sum())
print(df["disp4"].value_counts(normalize=True).round(3))

## Part 2. Regression: what will a stay cost, known at admission? *(Ethan and Niko)*
**The rule:** the model runs at admission, so it may only use what a hospital knows when the patient arrives. Diagnosis, procedure, severity, length of stay and discharge status are set later, so they are not allowed.

First, a Fisher-style F test on the train rows: how much of the difference in cost each admission-time variable explains.

In [ ]:
train, test = df[df['split'] == 'train'], df[df['split'] == 'test']

def f_test(t, col):
    y_cost = t['log_total_costs']; g = y_cost.groupby(t[col])
    ssb = (g.size() * (g.mean() - y_cost.mean()) ** 2).sum()
    k, n = g.ngroups, len(t); sst = ((y_cost - y_cost.mean()) ** 2).sum()
    return (ssb / (k - 1)) / ((sst - ssb) / (n - k)), ssb / sst

ADMISSION = ['age_group', 'type_of_admission', 'emergency_department_indicator', 'payment_typology_1',
             'health_service_area', 'permanent_facility_id', 'hospital_county', 'gender', 'race', 'ethnicity']
ranking = pd.DataFrame([(c, *f_test(train, c)) for c in ADMISSION], columns=['variable', 'F', 'share_of_cost_variance'])
print(ranking.sort_values('share_of_cost_variance', ascending=False).round(3).to_string(index=False))
print("\nFacility's F is low only because it has 204 groups; compare the share of variance explained.")

### The final regression model and its test score
Linear regression on four variables: age group, type of admission, main payer, facility. It is explainable, which a hospital needs.

We train and test on the 90% of stays under the high-cost cutoff. The cutoff uses the final cost, so a hospital cannot apply it at admission: this model describes typical stays.

In [ ]:
LEAN = ['age_group', 'type_of_admission', 'payment_typology_1', 'permanent_facility_id']
cutoff = df.loc[df['high_cost'] == 1, 'total_costs'].min()
trim_train, trim_test = train[train['high_cost'] == 0], test[test['high_cost'] == 0]
print(f"Cutoff ${int(cutoff):,}: {len(trim_train):,} train and {len(trim_test):,} test stays remain")

enc = OneHotEncoder(handle_unknown='ignore').fit(trim_train[LEAN])
final = LinearRegression().fit(enc.transform(trim_train[LEAN]), trim_train['log_total_costs'])

def report(name, t):
    p = final.predict(enc.transform(t[LEAN]))
    r2, typical, avg = r2_score(t['log_total_costs'], p), median_absolute_error(t['total_costs'], np.exp(p)), mean_absolute_error(t['total_costs'], np.exp(p))
    print(f"{name} ({len(t):,}): R2 (log) = {r2:.3f} | typical miss = ${typical:,.0f} | average miss = ${avg:,.0f}")
    return r2, typical, avg

typical_stays = report('FINAL model on typical test stays', trim_test)
all_stays = report('Same model on ALL test stays      ', test)

### What drives cost
Percent difference in cost against a patient who is 70 or older, admitted through the emergency department, with Medicare. Negative means cheaper.

In [ ]:
REF = {'age_group': '70 or Older', 'type_of_admission': 'Emergency', 'payment_typology_1': 'Medicare'}
Xd = pd.get_dummies(trim_train[LEAN]).astype(float)
lm = LinearRegression().fit(Xd, trim_train['log_total_costs'])
co = pd.Series(lm.coef_, index=Xd.columns)
rows = [(v, c[len(v) + 1:], round((np.exp(co[c] - co[v + '_' + REF[v]]) - 1) * 100, 1))
        for v in REF for c in co.index if c.startswith(v + '_') and c != v + '_' + REF[v]]
print(pd.DataFrame(rows, columns=['variable', 'group', 'cost_change_pct']).sort_values(['variable', 'cost_change_pct']).to_string(index=False))
common = trim_train['permanent_facility_id'].value_counts(); base = common.index[0]
fac = pd.Series({c[len('permanent_facility_id_'):]: (np.exp(co[c] - co['permanent_facility_id_' + base]) - 1) * 100
                 for c in co.index if c.startswith('permanent_facility_id_')})
fac = fac[[i for i in fac.index if common.get(i, 0) >= 300]].sort_values()
print(f"\nHospitals against the most common one ({base}): 3 lowest and 3 highest")
print(pd.concat([fac.head(3), fac.tail(3)]).round(1).to_string())

### Regression experiments behind the final model *(full mode only)*
Baseline, the variable screen (add one variable at a time), k-NN, the neural network and the decision tree picture. Set `RUN_FULL = True` at the top to run them.

In [ ]:
if RUN_FULL:
    # Prepare the data the way the original regression starter did
    df['log_length_of_stay'] = np.log(df['length_of_stay'])
    train = df[df['split'] == 'train'].reset_index(drop=True)
    y = train['log_total_costs']
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    from sklearn.dummy import DummyRegressor
    CATEGORICAL = ['age_group', 'type_of_admission', 'emergency_department_indicator', 'payment_typology_1', 'health_service_area', 'permanent_facility_id']
    NUMERIC = []

In [ ]:
if RUN_FULL:
    def score(model, cat, num, name, cv=cv):
        pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
        # out-of-fold predictions: each row is predicted by a model that did not see it
        pred = cross_val_predict(Pipeline([('pre', pre), ('m', model)]), train[cat + num], y, cv=cv)
        dollars_true, dollars_pred = np.exp(y), np.exp(pred)
        out = dict(name=name, R2_log=r2_score(y, pred),
                   median_miss=median_absolute_error(dollars_true, dollars_pred),
                   MAE=mean_absolute_error(dollars_true, dollars_pred))
        print(f"{name}: R2 (log) = {out['R2_log']:.3f} | median miss = ${out['median_miss']:,.0f} | MAE = ${out['MAE']:,.0f}")
        return out

    results = []
    results.append(score(DummyRegressor(strategy='mean'), CATEGORICAL, NUMERIC, 'Baseline (always the average)'))
    display(results.append(score(LinearRegression(), CATEGORICAL, NUMERIC, 'Linear regression')))

In [ ]:
if RUN_FULL:
    # Explainable model (group decision 11/3): four variables, linear regression on log cost. Scored the same way as above.
    LEAN = ['age_group', 'type_of_admission', 'payment_typology_1', 'permanent_facility_id']
    display(results.append(score(LinearRegression(), LEAN, [], 'Lean linear regression (4 variables)')))

In [ ]:
if RUN_FULL:
    STARTER = ['age_group', 'type_of_admission', 'emergency_department_indicator', 'payment_typology_1', 'health_service_area']
    CANDIDATES = ['permanent_facility_id', 'hospital_county', 'zip_code', 'payment_typology_2', 'payment_typology_3',
                  'gender', 'race', 'ethnicity', 'patient_disposition', 'apr_severity_of_illness_code',
                  'apr_risk_of_mortality', 'apr_drg_code', 'ccsr_diagnosis_code', 'ccsr_procedure_code', 'log_length_of_stay']

    cv3 = KFold(n_splits=3, shuffle=True, random_state=42)   # 3 folds so the screen runs faster
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):          # hide the per-run printouts
        base = score(LinearRegression(), STARTER, [], 'starter', cv=cv3)
        rows = []
        for c in CANDIDATES:
            cat = STARTER + ([] if c == 'log_length_of_stay' else [c])
            num = ['log_length_of_stay'] if c == 'log_length_of_stay' else []
            r = score(LinearRegression(), cat, num, c, cv=cv3)
            rows.append((c, r['R2_log'] - base['R2_log'], r['median_miss'], r['MAE']))
    screen = pd.DataFrame(rows, columns=['added', 'R2_gain', 'median_miss', 'MAE']).sort_values('R2_gain', ascending=False)
    print(f"starter: R2 {base['R2_log']:.3f}, median miss ${base['median_miss']:,.0f}, MAE ${base['MAE']:,.0f}")
    display(screen.round(3))

In [ ]:
if RUN_FULL:
    # Step 3b: k-NN regression on the admission-time variables. k-NN is slow, so it uses a 60,000-row sample of the train rows.
    from sklearn.neighbors import KNeighborsRegressor
    sub = train.sample(60000, random_state=42).reset_index(drop=True)
    y_sub = sub['log_total_costs']
    def knn_score(k): pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)]); pred = cross_val_predict(Pipeline([('pre', pre), ('m', KNeighborsRegressor(n_neighbors=k, n_jobs=-1))]), sub[CATEGORICAL], y_sub, cv=cv3); return dict(k=k, R2_log=r2_score(y_sub, pred), median_miss=median_absolute_error(np.exp(y_sub), np.exp(pred)), MAE=mean_absolute_error(np.exp(y_sub), np.exp(pred)))
    knn_results = pd.DataFrame([knn_score(k) for k in [5, 10, 25, 50, 100]])
    display(knn_results.round(3))

In [ ]:
if RUN_FULL:
    # Step 3c: multilayer perceptron (week 7) on the admission-time variables. 40,000-row sample of the train rows, 3-fold cross-validation. Linear regression is run on the same sample so the comparison is fair.
    from sklearn.neural_network import MLPRegressor
    import matplotlib.pyplot as plt
    samp = train.sample(40000, random_state=42).reset_index(drop=True)
    y_s = samp['log_total_costs']
    def run_sample(name, model): pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)]); pred = cross_val_predict(Pipeline([('pre', pre), ('m', model)]), samp[CATEGORICAL], y_s, cv=cv3); return dict(model=name, R2_log=r2_score(y_s, pred), median_miss=median_absolute_error(np.exp(y_s), np.exp(pred)), MAE=mean_absolute_error(np.exp(y_s), np.exp(pred)))
    mlp_results = pd.DataFrame([run_sample('Linear regression (same sample)', LinearRegression())] + [run_sample(f'MLP {h}', MLPRegressor(hidden_layer_sizes=h, early_stopping=True, max_iter=100, random_state=0)) for h in [(32,), (64,), (64, 32)]])
    display(mlp_results.round(3))

In [ ]:
if RUN_FULL:
    # Loss curve (clean version): center the target so training starts near zero, and show the validation score too.
    y_c = y_s - y_s.mean()
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)])
    best = Pipeline([('pre', pre), ('m', MLPRegressor(hidden_layer_sizes=(64,), early_stopping=True, max_iter=200, random_state=0))]).fit(samp[CATEGORICAL], y_c)
    m = best.named_steps['m']
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(m.loss_curve_); ax[0].set_title('Training loss'); ax[0].set_xlabel('Training round (epoch)'); ax[0].set_ylabel('Loss (centered target)')
    ax[1].plot(m.validation_scores_); ax[1].set_title('Validation R-squared'); ax[1].set_xlabel('Training round (epoch)'); ax[1].set_ylabel('R-squared')
    plt.suptitle('Multilayer perceptron (64 nodes)'); plt.show()

In [ ]:
if RUN_FULL:
    # Shallow decision tree (depth 3) as a picture of what drives cost. Facility is left out here because it has 200+ values; the tree is for the story, the linear model above is for predictions.
    import matplotlib.pyplot as plt
    from sklearn.tree import DecisionTreeRegressor, plot_tree
    TREE_VARS = ['age_group', 'type_of_admission', 'payment_typology_1']
    Xt = pd.get_dummies(train[TREE_VARS]).astype(float)
    tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=500, random_state=0).fit(Xt, y)
    plt.figure(figsize=(16, 7)); plot_tree(tree, feature_names=list(Xt.columns), filled=True, rounded=True, fontsize=8); plt.title('Decision tree, depth 3 (value = average log cost)'); plt.show()

## Part 3. Classification: where does the patient go next? *(Mac and Jacob)*
**Question:** early in a hospital stay, can we predict where the patient goes next? Four classes: Home, Post-acute facility, Died or hospice, Other.

**Rules the group followed**
1. Every choice (features, tree depth, settings) was made by 5-fold cross-validation on the TRAIN rows only.
2. The 40,000 test rows are scored once, for the final models only.
3. Length of stay, severity, risk of mortality, DRG, procedure, charges and costs are excluded because they are only known at or after discharge. They appear only in clearly labeled CEILING runs.
4. Metric: macro F1 (every class counts equally), with per-class recall and precision, always compared with a majority-class baseline.

In [ ]:
# Train rows only for every choice. Test rows stay locked until Part 8.
train = df[df["split"] == "train"].dropna(subset=["disp4"]).copy()
y = train["disp4"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

LABELS = ["Home", "Post-acute facility", "Died or hospice", "Other"]
CLASSES = np.sort(y.unique())                       # alphabetical, the order sklearn uses
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}    # integer labels, needed for the neural net
PRIOR = y.value_counts(normalize=True)[CLASSES].values

CAT_FEATURES = ["age_group", "gender", "race", "ethnicity", "type_of_admission",
                "emergency_department_indicator", "payment_typology_1",
                "apr_mdc_code", "permanent_facility_id"]
with_dx = CAT_FEATURES + ["ccsr_diagnosis_code"]
no_sens = [c for c in with_dx if c not in ["race", "ethnicity", "gender"]]

LOG = []
def one_hot(cols):
    # codes seen fewer than 50 times are pooled into one "rare" level
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cols)])

def record(name, model_name, pred, n_feat):
    p, r, _, _ = precision_recall_fscore_support(y, pred, labels=LABELS, zero_division=0)
    row = {"experiment": name, "model": model_name, "n_features": n_feat,
           "macro_f1": round(f1_score(y, pred, average="macro", zero_division=0), 4)}
    for lab, pi, ri in zip(LABELS, p, r):
        row[f"recall_{lab}"], row[f"precision_{lab}"] = round(ri, 3), round(pi, 3)
    LOG.append(row)
    pd.DataFrame(LOG).to_csv("experiments_log.csv", index=False)
    print(f"{name} | {model_name} | macro F1 {row['macro_f1']}")

def run_experiment(name, cols, clf):
    pipe = Pipeline([("pre", one_hot(cols)), ("clf", clf)])
    pred = cross_val_predict(pipe, train[cols], y, cv=cv, n_jobs=-1)
    record(name, type(clf).__name__, pred, len(cols))

def run_mlp(name, cols, hidden=(128, 64)):
    # MLPClassifier has no class_weight. Train normally, then divide each predicted
    # probability by the class share (prior). Same effect as balanced weights.
    pipe = Pipeline([("pre", one_hot(cols)),
                     ("clf", MLPClassifier(hidden_layer_sizes=hidden, early_stopping=True,
                                           max_iter=100, random_state=42))])
    proba = cross_val_predict(pipe, train[cols], y.map(CLASS_ID), cv=cv,
                              method="predict_proba", n_jobs=-1)
    record(name, "MLPClassifier", CLASSES[np.argmax(proba / PRIOR, axis=1)], len(cols))

def LR(C=1.0): return LogisticRegression(C=C, class_weight="balanced", max_iter=500)
def DT(depth): return DecisionTreeClassifier(max_depth=depth, class_weight="balanced",
                                             min_samples_leaf=50, random_state=42)

### The final classification models, scored once on the test rows
Final variables, chosen by cross-validation in the experiments below: age group, type of admission, main payer, facility and the **diagnosis code**.

**Note:** the diagnosis code is assigned when the stay is coded, so it is not known at admission. The cost model above never uses it. Part 4 therefore uses a classifier that sees admission-time variables only, and the next cell shows what that costs.

In [ ]:
FINAL_FEATS = ["age_group", "type_of_admission", "payment_typology_1", "permanent_facility_id", "ccsr_diagnosis_code"]
test_rows = df[df["split"] == "test"].dropna(subset=["disp4"])
print("FINAL FEATURES:", FINAL_FEATS)

In [ ]:
test_df = test_rows
yt = test_df["disp4"]
Xtr, Xte = train[FINAL_FEATS], test_df[FINAL_FEATS]

preds = {"Majority baseline": np.array(["Home"] * len(yt))}
for name, clf in {"Decision tree (depth 20)": DT(20), "Logistic regression": LR()}.items():
    preds[name] = Pipeline([("pre", one_hot(FINAL_FEATS)), ("clf", clf)]).fit(Xtr, y).predict(Xte)

mlp = Pipeline([("pre", one_hot(FINAL_FEATS)),
                ("clf", MLPClassifier(hidden_layer_sizes=(128, 64), early_stopping=True,
                                      max_iter=100, random_state=42))])
mlp.fit(Xtr, y.map(CLASS_ID))
preds["MLP (128, 64)"] = CLASSES[np.argmax(mlp.predict_proba(Xte) / PRIOR, axis=1)]

rows = []
for name, pr in preds.items():
    p_, r_, _, _ = precision_recall_fscore_support(yt, pr, labels=LABELS, zero_division=0)
    row = {"model": name, "macro_f1": round(f1_score(yt, pr, average="macro", zero_division=0), 4)}
    for lab, pi, ri in zip(LABELS, p_, r_):
        row[f"recall_{lab}"], row[f"precision_{lab}"] = round(ri, 3), round(pi, 3)
    rows.append(row)
results_test = pd.DataFrame(rows)
results_test.to_csv("test_results.csv", index=False)
pd.DataFrame({"row_id": test_df.index, "actual": yt.values, **{k: v for k, v in preds.items()}}
             ).to_csv("test_predictions.csv", index=False)
results_test

In [ ]:
# Logistic regression (our best model) in full, and how much better than random each flag is
print(classification_report(yt, preds["Logistic regression"], digits=3, zero_division=0))
base_rate = yt.value_counts(normalize=True)
lift = pd.DataFrame({"precision": [results_test.set_index("model").loc["Logistic regression", f"precision_{l}"] for l in LABELS],
                     "base_rate": [base_rate[l] for l in LABELS]}, index=LABELS)
lift["lift"] = (lift["precision"] / lift["base_rate"]).round(2)
lift.round(3)

In [ ]:
# The same logistic regression with admission-time variables only (no diagnosis code): the version the link in Part 4 can use
LINK_FEATS = ["age_group", "type_of_admission", "payment_typology_1", "permanent_facility_id"]
adm = Pipeline([("pre", one_hot(LINK_FEATS)), ("clf", LR())]).fit(train[LINK_FEATS], y)
adm_pred = adm.predict(test_rows[LINK_FEATS])
print(f"Logistic regression, admission-time variables only: test macro F1 = {f1_score(test_rows['disp4'], adm_pred, average='macro'):.3f}")
print("With the diagnosis code it scores 0.379; the majority baseline scores 0.221.")

## Part 9. Results and interpretation
**Official test macro F1 (40,000 rows, scored once):** majority baseline 0.221, decision tree 0.359, neural network 0.359, **logistic regression 0.379 (best)**.

- **Signal is real but modest.** For the three non-Home classes, a flag from the logistic regression is about 2.6 to 4.3 times more likely to be correct than the class's base rate (the lift table), but most flags are false alarms, especially for Died or hospice and Other.
- **Why logistic regression wins.** It had the best cross-validation score among the simple models and held up on the test rows. The neural network scored highest in cross-validation (0.385) but fell to 0.359 on the test rows, which suggests its CV score was optimistic. We therefore use logistic regression for the link model.
- **The limit is the information, not the tuning.** Even with discharge-only variables (length of stay, severity, mortality risk) the best CV macro F1 was 0.44. Billing records have no information on functional ability, home support, bed availability or patient and family choices, which drive many discharge decisions.
- **Fairness.** Dropping race, ethnicity and gender cost only about 0.002 macro F1, but this does not make the model fair: facility, payer and diagnosis can act as proxies. Per-group results are handled by the fairness analysis.
- **Limitations.** The 4-class grouping is a proposal (Other mixes unrelated destinations). Predictions assume the information is known early in the stay. Fisher scores are only a rough guide for categorical data.

### Classification experiments behind the final models *(full mode only)*
Baselines, tuning, feature selection (Fisher score, tree importances, drop-one ablation), final candidates and the probability exports. Set `RUN_FULL = True` at the top to run them. Total run time is roughly 20 to 30 minutes, mostly the cross-validation runs.

In [ ]:
if RUN_FULL:
    run_experiment("majority baseline (always Home)", CAT_FEATURES, DummyClassifier(strategy="most_frequent"))
    run_experiment("early spec, 9 vars", CAT_FEATURES, DT(8))
    run_experiment("early spec, 9 vars", CAT_FEATURES, LR())
    run_experiment("early spec + diagnosis", with_dx, DT(8))
    display(run_experiment("early spec + diagnosis", with_dx, LR()))

In [ ]:
if RUN_FULL:
    # CEILING runs: discharge-only information. For comparison only.
    ceil_cat = with_dx + ["apr_severity_of_illness_code", "apr_risk_of_mortality"]
    los = ["length_of_stay"]
    for nm, cats, nums in [("CEILING + length_of_stay", with_dx, los),
                           ("CEILING + length_of_stay + severity + mortality risk", ceil_cat, los)]:
        pipe = Pipeline([("pre", ColumnTransformer([
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), cats),
            ("num", StandardScaler(), nums)])), ("clf", LR())])
        pred = cross_val_predict(pipe, train[cats + nums], y, cv=cv, n_jobs=-1)
        record(nm, "LogisticRegression", pred, len(cats) + len(nums))

In [ ]:
if RUN_FULL:
    # Tune the tree depth and the logistic regression penalty
    for d in [6, 10, 14, 20, 30, 40]:
        run_experiment(f"early spec + diagnosis, tree depth {d}", with_dx, DT(d))
    for c in [0.1, 10]:
        run_experiment(f"early spec + diagnosis, LR C={c}", with_dx, LR(c))
    display(pd.DataFrame(LOG).sort_values("macro_f1", ascending=False)[["experiment", "model", "n_features", "macro_f1"]])

In [ ]:
if RUN_FULL:
    # A. Fisher score on train rows, per one-hot column, then rolled up to the original variable
    pre_f = one_hot(with_dx)
    Xs = sparse.csr_matrix(pre_f.fit_transform(train[with_dx]))
    names = [n.replace("cat__", "", 1) for n in pre_f.get_feature_names_out()]
    yv = y.values
    mu = np.asarray(Xs.mean(axis=0)).ravel()
    between, within = np.zeros(Xs.shape[1]), np.zeros(Xs.shape[1])
    for k in np.unique(yv):
        sub = Xs[yv == k]
        m_k = np.asarray(sub.mean(axis=0)).ravel()
        v_k = np.asarray(sub.multiply(sub).mean(axis=0)).ravel() - m_k ** 2
        between += sub.shape[0] * (m_k - mu) ** 2
        within += sub.shape[0] * v_k
    fisher = between / np.where(within == 0, np.nan, within)

    def to_variable(col):
        hits = [f for f in with_dx if col.startswith(f + "_")]
        return max(hits, key=len) if hits else col

    col_df = pd.DataFrame({"column": names, "fisher": fisher})
    col_df["variable"] = col_df["column"].map(to_variable)
    fisher_rank = col_df.groupby("variable")["fisher"].agg(fisher_max="max", fisher_mean="mean")
    display(fisher_rank.sort_values("fisher_max", ascending=False).round(4))

In [ ]:
if RUN_FULL:
    # B. Decision tree importances (second opinion)
    tree_pipe = Pipeline([("pre", pre_f), ("clf", DT(8))]).fit(train[with_dx], y)
    imp = pd.Series(tree_pipe.named_steps["clf"].feature_importances_, index=names)
    tree_rank = imp.groupby(imp.index.map(to_variable)).sum().rename("tree_importance")
    compare = fisher_rank.join(tree_rank)
    compare["fisher_rank"] = compare["fisher_max"].rank(ascending=False)
    compare["tree_rank"] = compare["tree_importance"].rank(ascending=False)
    display(compare.sort_values("tree_rank").round(4))

In [ ]:
if RUN_FULL:
    # C. Drop-one ablation: remove each variable, rerun logistic regression, see what it costs
    run_experiment("ablation baseline: all 10 vars", with_dx, LR())
    for v in with_dx:
        run_experiment(f"ablate {v}", [c for c in with_dx if c != v], LR())
    full_f1 = [r for r in LOG if r["experiment"] == "ablation baseline: all 10 vars"][0]["macro_f1"]
    abl = pd.DataFrame([r for r in LOG if r["experiment"].startswith("ablate ")])
    abl["change_vs_full"] = (abl["macro_f1"] - full_f1).round(4)
    display(abl.sort_values("macro_f1")[["experiment", "macro_f1", "change_vs_full"]])

In [ ]:
if RUN_FULL:
    cands = {
        "A: 7 vars (no race/ethnicity/gender)": no_sens,
        "B: A minus ED indicator": [c for c in no_sens if c != "emergency_department_indicator"],
        "C: B minus MDC group": [c for c in no_sens if c not in ["emergency_department_indicator", "apr_mdc_code"]],
        "D: age, payer, facility, diagnosis": ["age_group", "payment_typology_1",
                                               "permanent_facility_id", "ccsr_diagnosis_code"],
    }
    for nm, feats in cands.items():
        run_experiment(f"final candidate {nm}", feats, LR())

    FINAL_FEATS = ["age_group", "type_of_admission", "payment_typology_1",
                   "permanent_facility_id", "ccsr_diagnosis_code"]
    print("\nFINAL FEATURES:", FINAL_FEATS)

In [ ]:
if RUN_FULL:
    run_mlp("MLP (64) on 7 vars, prior-corrected", no_sens, (64,))
    display(run_mlp("MLP (128, 64) on 7 vars, prior-corrected", no_sens, (128, 64)))

In [ ]:
if RUN_FULL:
    run_experiment("FINAL 5 vars", FINAL_FEATS, DT(20))       # depth 20 was the CV peak in Part 3
    run_experiment("FINAL 5 vars", FINAL_FEATS, LR())
    run_mlp("FINAL 5 vars MLP (128, 64)", FINAL_FEATS, (128, 64))
    display(pd.DataFrame(LOG).query("experiment.str.contains('FINAL 5')")[["experiment", "model", "macro_f1"]])

In [ ]:
if RUN_FULL:
    pipe = Pipeline([("pre", one_hot(FINAL_FEATS)), ("clf", LR())])
    oof = cross_val_predict(pipe, train[FINAL_FEATS], y, cv=cv, method="predict_proba", n_jobs=-1)
    pipe.fit(train[FINAL_FEATS], y)
    cols = ["p_" + c.replace(" ", "_") for c in pipe.named_steps["clf"].classes_]
    test_rows = df[df["split"] == "test"].dropna(subset=["disp4"])
    test_proba = pipe.predict_proba(test_rows[FINAL_FEATS])

    out = pd.concat([pd.DataFrame(oof, columns=cols, index=train.index),
                     pd.DataFrame(test_proba, columns=cols, index=test_rows.index)]).sort_index()
    out.insert(0, "split", df.loc[out.index, "split"].values)
    out.index.name = "row_id"
    out.to_csv("oof_probabilities.csv")
    print(out.shape, "| row sums:", out[cols].sum(axis=1).round(6).unique()[:3])
    display(out.groupby("split")[cols].mean().round(3))

In [ ]:
if RUN_FULL:
    LINK_FEATS = ["age_group", "type_of_admission", "payment_typology_1", "permanent_facility_id"]
    run_experiment("LINK export vars (admission-time only)", LINK_FEATS, LR())

    pipe = Pipeline([("pre", one_hot(LINK_FEATS)), ("clf", LR())])
    oof = cross_val_predict(pipe, train[LINK_FEATS], y, cv=cv, method="predict_proba", n_jobs=-1)
    pipe.fit(train[LINK_FEATS], y)
    cols = ["p_" + c.replace(" ", "_") for c in pipe.named_steps["clf"].classes_]
    test_p = pipe.predict_proba(test_rows[LINK_FEATS])
    out = pd.concat([pd.DataFrame(oof, columns=cols, index=train.index),
                     pd.DataFrame(test_p, columns=cols, index=test_rows.index)]).sort_index()
    out.insert(0, "split", df.loc[out.index, "split"].values)
    out.index.name = "row_id"
    out.to_csv("oof_probabilities_admission.csv")
    print(out.shape)

## Part 4. The link: does knowing the likely destination help predict cost?
The classifier predicts where the patient goes after the stay. We test three versions of the cost model on the same typical stays:
- **A:** the four admission variables only (the final cost model).
- **B:** A plus the **predicted** destination, from a logistic regression on the same admission-time variables (no diagnosis code). This is the link a hospital could use at admission.
- **C:** A plus the **true** destination. A hospital only knows it after the stay, so this shows the most the destination could add.

The destination probabilities for the training rows are out-of-fold: each row is scored by a model that did not train on it. We use a plain (unweighted) logistic regression here so the probabilities are true rates, not inflated for the rare classes.

In [ ]:
tr, te = df[(df['split'] == 'train') & (df['high_cost'] == 0)], df[(df['split'] == 'test') & (df['high_cost'] == 0)]
enc_link = OneHotEncoder(handle_unknown='ignore').fit(tr[LEAN])
Xtr, Xte = enc_link.transform(tr[LEAN]), enc_link.transform(te[LEAN])
cv_link = KFold(5, shuffle=True, random_state=42)

clf = LogisticRegression(max_iter=300)
P_tr = cross_val_predict(clf, Xtr, tr['disp4'], cv=cv_link, method='predict_proba')
clf.fit(Xtr, tr['disp4']); P_te = clf.predict_proba(Xte)
acc = accuracy_score(te['disp4'], clf.classes_[P_te.argmax(1)])
majority = (te['disp4'] == tr['disp4'].value_counts().idxmax()).mean()
print(f"Destination classifier on typical test stays: accuracy {acc:.3f} against {majority:.3f} for always guessing Home")

In [ ]:
def cost_model(name, A_tr, A_te):
    m = LinearRegression().fit(A_tr, tr['log_total_costs']); p = m.predict(A_te)
    r2 = r2_score(te['log_total_costs'], p); typical = median_absolute_error(te['total_costs'], np.exp(p)); avg = mean_absolute_error(te['total_costs'], np.exp(p))
    print(f"{name}: R2 (log) = {r2:.4f} | typical miss = ${typical:,.0f} | average miss = ${avg:,.0f}")
    return r2

true_tr = sparse.csr_matrix(pd.get_dummies(tr['disp4']).reindex(columns=clf.classes_, fill_value=0).values.astype(float))
true_te = sparse.csr_matrix(pd.get_dummies(te['disp4']).reindex(columns=clf.classes_, fill_value=0).values.astype(float))
r2_A = cost_model('A  admission variables only              ', Xtr, Xte)
r2_B = cost_model('B  + PREDICTED destination (the link)    ', sparse.hstack([Xtr, sparse.csr_matrix(P_tr)]).tocsr(), sparse.hstack([Xte, sparse.csr_matrix(P_te)]).tocsr())
r2_C = cost_model('C  + TRUE destination (known only later) ', sparse.hstack([Xtr, true_tr]).tocsr(), sparse.hstack([Xte, true_te]).tocsr())
print(f"\nPredicted destination adds {r2_B - r2_A:+.4f} to R2. True destination adds {r2_C - r2_A:+.4f}.")

**Mac's own export.** The classification group exported the same destination probabilities from their notebook (admission-time variables, out-of-fold for the train rows). Using their file instead of ours:

In [ ]:
MAC_URL = 'https://raw.githubusercontent.com/ethanmooseg-star/bte413-hospital-cost/main/oof_probabilities_admission.csv.gz'
mac = pd.read_csv(MAC_URL).set_index('row_id')
assert (mac['split'].values == df['split'].values).all(), "Mac's file does not line up with the data"
mac_cols = ['p_Died_or_hospice', 'p_Home', 'p_Other', 'p_Post-acute_facility']      # same class order as the classifier above
r2_Bm = cost_model("B' + Mac's exported destination probabilities  ", sparse.hstack([Xtr, sparse.csr_matrix(mac.loc[tr.index, mac_cols].values)]).tocsr(),
                   sparse.hstack([Xte, sparse.csr_matrix(mac.loc[te.index, mac_cols].values)]).tocsr())
print(f"\nWith Mac's file the predicted destination adds {r2_Bm - r2_A:+.4f} to R2: the same answer.")

**Reading it:** admission information barely predicts where a patient goes (accuracy is no better than always guessing Home), so the predicted destination adds almost nothing to the cost model. Even the true destination adds only a little. The two models are better used side by side: one estimates cost, the other helps plan discharge. The destination does not improve the cost estimate.

## Part 5. Do the numbers match the slides?

In [ ]:
scores = results_test.set_index('model')['macro_f1']
checks = [
    ('Regression: final model R2, typical stays', typical_stays[0], 0.393, 0.001),
    ('Regression: typical miss ($)', typical_stays[1], 5005, 5),
    ('Regression: average miss ($)', typical_stays[2], 7571, 5),
    ('Regression: same model R2, all test stays', all_stays[0], 0.312, 0.001),
    ('Regression: same model average miss ($)', all_stays[2], 17148, 5),
    ('Classification: majority baseline macro F1', scores['Majority baseline'], 0.221, 0.001),
    ('Classification: decision tree macro F1', scores['Decision tree (depth 20)'], 0.359, 0.001),
    ('Classification: neural network macro F1', scores['MLP (128, 64)'], 0.359, 0.001),
    ('Classification: logistic regression macro F1', scores['Logistic regression'], 0.379, 0.001),
    ('Link: R2 of cost model A (admission only)', r2_A, 0.393, 0.001),
]
for name, got, want, tol in checks:
    print(f"{'OK     ' if abs(got - want) <= tol else 'DIFFERS'}  {name}: {got:,.3f} (slides: {want:,})")
print(f"{'OK     ' if r2_B - r2_A < 0.002 else 'DIFFERS'}  Link: predicted destination adds {r2_B - r2_A:+.4f} to R2 (slides: about nothing)")
print(f"{'OK     ' if r2_Bm - r2_A < 0.002 else 'DIFFERS'}  Link with Mac's export: adds {r2_Bm - r2_A:+.4f} to R2 (slides: about nothing)")